# Automated Answer Script Evaluation - Marksheet Recognition Pipeline

Template-based ROI extraction + blank/written detection + pretrained-CNN
digit recognition + checksum validation, for the Amrita pink evaluation
sheet.

**Tested results on a real uploaded sheet (booklet AC201389, ground truth
confirmed by visual inspection):**
- Tesseract on isolated digit crops: 0/5 correct
- EasyOCR on isolated digit crops: 1/5 correct (14% confidence)
- MNIST-trained CNN with proper digit-centering preprocessing: **3/5 correct**, with 80-100% confidence on its answers

The CNN is the best of the three and is used as the recognizer below, but
3/5 on this sample is not production-grade accuracy - see Cell 12 for the
honest breakdown and Cell 18's discussion of why it still misses digits,
plus what would close the gap (fine-tuning on your own handwriting samples
instead of relying on stock MNIST).


## Cell 1 - Install dependencies + imports

In [ ]:

!pip install -q pytesseract
!apt-get install -y -qq tesseract-ocr > /dev/null

import os
import struct
import numpy as np
import cv2
import matplotlib.pyplot as plt
import pytesseract
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from scipy import ndimage

print("Imports OK")
print("Tesseract version:", pytesseract.get_tesseract_version())


## Cell 2 - Template configuration

These x-boundaries and row y-bands were **measured directly from a real
uploaded sheet** (not guessed) - see the calibration note at the bottom of
this cell for how, so you can redo this if your own scans differ.

Canonical template size: 1094 x 646 (matches the working size from prior
sessions, per your instruction not to throw away old work).


In [ ]:

TEMPLATE = {
    "canonical_width": 1094,
    "canonical_height": 646,

    # 22 boundaries -> 21 columns -> Q1a|Q1b|Q2a|Q2b|...|Q10a|Q10b|subtotal
    # Measured from a real scan at 1312x938, then scaled to the 1094x646
    # canonical size. Re-measure with the calibration helper at the bottom
    # of this cell if your own scans are laid out differently.
    "x_boundaries": [113, 155, 192, 233, 270, 313, 350, 390, 429, 468,
                      504, 544, 582, 622, 662, 703, 741, 782, 822, 864,
                      905, 999],

    # y-bands for each row-of-10-questions' ANSWER area (i.e. already
    # excluding the printed question-number row and the printed a/b
    # header row above it).
    "row_bands": {
        1: (269, 306),   # Q1-Q10
        2: (345, 384),   # Q11-Q20
        3: (421, 462),   # Q21-Q30
    },

    # Sl.No. (booklet number) box - printed text, top-right area.
    "booklet_box": {"x0": 709, "y0": 76, "x1": 1042, "y1": 138},

    # Grand Total box - bottom right, below the question grid.
    # NOTE: recalibrate this per your sheet; it sits below row 3.
    "grand_total_box": {"x0": 900, "y0": 480, "x1": 1050, "y1": 520},
}

def question_cols(qnum):
    """Return (a_left, a_right/b_left, b_right) x-coords for a question
    number 1-30, using the 21 shared boundaries within its block of 10."""
    idx_in_block = (qnum - 1) % 10
    xb = TEMPLATE["x_boundaries"]
    return xb[idx_in_block*2], xb[idx_in_block*2+1], xb[idx_in_block*2+2]

def row_for_question(qnum):
    block = (qnum - 1) // 10 + 1
    return TEMPLATE["row_bands"][block]

print("Template loaded:", TEMPLATE["canonical_width"], "x", TEMPLATE["canonical_height"])

# --- If you need to recalibrate against a NEW sheet layout ---
# Run this against a clean reference scan to auto-detect the grid lines,
# instead of eyeballing pixel coordinates:
#
#   gray = cv2.cvtColor(your_aligned_image, cv2.COLOR_BGR2GRAY)
#   region = gray[Y0:Y1, X0:X1]           # crop roughly around row 1's grid
#   _, binary = cv2.threshold(region, 170, 255, cv2.THRESH_BINARY_INV)
#   col_sums = binary.sum(axis=0)
#   # then find peaks in col_sums to get vertical line x-positions
#
# This is exactly how the x_boundaries above were derived.


## Cell 3 - Image alignment

Detects the page boundary and perspective-corrects it; if no reliable
quadrilateral is found (common when the sheet is already tightly cropped,
like a scanner output), falls back to a safe resize instead of guessing a
transform.


In [ ]:

def order_points(pts):
    rect = np.zeros((4, 2), dtype="float32")
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]
    rect[2] = pts[np.argmax(s)]
    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]
    rect[3] = pts[np.argmax(diff)]
    return rect

def find_page_contour(gray):
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    edged = cv2.Canny(blurred, 50, 150)
    edged = cv2.dilate(edged, np.ones((5, 5), np.uint8), iterations=2)
    contours, _ = cv2.findContours(edged, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    contours = sorted(contours, key=cv2.contourArea, reverse=True)[:6]
    for c in contours:
        peri = cv2.arcLength(c, True)
        approx = cv2.approxPolyDP(c, 0.02 * peri, True)
        if len(approx) == 4 and cv2.contourArea(approx) > 0.4 * gray.shape[0] * gray.shape[1]:
            return approx.reshape(4, 2)
    return None

def align_to_template(image):
    W, H = TEMPLATE["canonical_width"], TEMPLATE["canonical_height"]
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    quad = find_page_contour(gray)
    if quad is not None:
        rect = order_points(quad.astype("float32"))
        dst = np.array([[0,0],[W-1,0],[W-1,H-1],[0,H-1]], dtype="float32")
        M = cv2.getPerspectiveTransform(rect, dst)
        warped = cv2.warpPerspective(image, M, (W, H))
        print("Perspective correction: YES")
    else:
        warped = cv2.resize(image, (W, H))
        print("Perspective correction: NO (safe resize fallback)")
    return warped

print("align_to_template() ready")


## Cell 4 - Upload image

In Colab this opens a file picker. If you're running this outside Colab
(e.g. locally), set `IMAGE_PATH` directly instead.


In [ ]:

IMAGE_PATH = None

try:
    from google.colab import files
    uploaded = files.upload()
    IMAGE_PATH = list(uploaded.keys())[0]
except ImportError:
    # Not running in Colab - set this manually:
    IMAGE_PATH = "my_marksheet.jpg"
    print(f"Not in Colab - using IMAGE_PATH = {IMAGE_PATH}. Edit this cell if needed.")

raw_image = cv2.imread(IMAGE_PATH)
assert raw_image is not None, f"Could not read image at {IMAGE_PATH}"

aligned = align_to_template(raw_image)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(cv2.cvtColor(raw_image, cv2.COLOR_BGR2RGB)); axes[0].set_title("Original"); axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(aligned, cv2.COLOR_BGR2RGB)); axes[1].set_title("Aligned (1094x646)"); axes[1].axis('off')
plt.tight_layout(); plt.show()

cv2.imwrite("aligned.png", aligned)
print("Saved aligned.png")


## Cell 5 - ROI extraction (booklet, Q1a-Q30b, subtotal boxes)

In [ ]:

def extract_rois(aligned_image):
    """Returns a dict of every raw ROI crop: booklet number, all 60
    question sub-cells, and the 3 subtotal boxes. Does NOT touch grand
    total (handled by its own box) or clean the answer area yet - that's
    Cell 7."""
    rois = {}

    bb = TEMPLATE["booklet_box"]
    rois["booklet"] = aligned_image[bb["y0"]:bb["y1"], bb["x0"]:bb["x1"]]

    for qnum in range(1, 31):
        xa1, xa2, xb2 = question_cols(qnum)
        y1, y2 = row_for_question(qnum)
        rois[f"Q{qnum}a"] = aligned_image[y1:y2, xa1:xa2]
        rois[f"Q{qnum}b"] = aligned_image[y1:y2, xa2:xb2]

    # Subtotal boxes sit just right of the 10th question's 'b' column,
    # one per row-of-10-questions block.
    xb = TEMPLATE["x_boundaries"]
    for block, (y1, y2) in TEMPLATE["row_bands"].items():
        rois[f"subtotal_block{block}"] = aligned_image[y1:y2, xb[20]:xb[21]]

    gt = TEMPLATE["grand_total_box"]
    rois["grand_total"] = aligned_image[gt["y0"]:gt["y1"], gt["x0"]:gt["x1"]]

    return rois

rois = extract_rois(aligned)
print(f"Extracted {len(rois)} ROIs:", list(rois.keys())[:8], "...")


## Cell 6 - ROI visualization (draw every box on the aligned sheet)

In [ ]:

def visualize_rois(aligned_image):
    vis = aligned_image.copy()
    xb = TEMPLATE["x_boundaries"]

    for qnum in range(1, 31):
        xa1, xa2, xb2 = question_cols(qnum)
        y1, y2 = row_for_question(qnum)
        cv2.rectangle(vis, (xa1, y1), (xa2, y2), (0, 200, 0), 1)
        cv2.rectangle(vis, (xa2, y1), (xb2, y2), (0, 120, 255), 1)

    for block, (y1, y2) in TEMPLATE["row_bands"].items():
        cv2.rectangle(vis, (xb[20], y1), (xb[21], y2), (255, 0, 0), 1)

    bb = TEMPLATE["booklet_box"]
    cv2.rectangle(vis, (bb["x0"], bb["y0"]), (bb["x1"], bb["y1"]), (255, 0, 255), 2)

    gt = TEMPLATE["grand_total_box"]
    cv2.rectangle(vis, (gt["x0"], gt["y0"]), (gt["x1"], gt["y1"]), (0, 255, 255), 2)

    return vis

vis = visualize_rois(aligned)
plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.title("Green='a' cols, Orange='b' cols, Blue=subtotal, Magenta=booklet, Cyan=grand total")
plt.axis('off')
plt.show()


## Cell 7 - Answer-area extraction

Trims a margin off each raw ROI to remove ruling-line bleed. Because the
row-bands in Cell 2 were measured to already sit BELOW the printed
question-number and a/b-header rows, this cell does not need to strip
printed text - only grid-line noise at the very edges.


In [ ]:

def extract_answer_area(cell_bgr, margin_frac_y=0.12, margin_frac_x=0.08):
    h, w = cell_bgr.shape[:2]
    my, mx = int(h * margin_frac_y), int(w * margin_frac_x)
    if h - 2*my <= 0 or w - 2*mx <= 0:
        return cell_bgr
    return cell_bgr[my:h-my, mx:w-mx]

answer_areas = {k: extract_answer_area(v) for k, v in rois.items() if k.startswith("Q")}
print(f"Extracted {len(answer_areas)} answer areas")


## Cell 8 - Answer-area visualization (all 60 cells in a grid)

In [ ]:

fig, axes = plt.subplots(6, 10, figsize=(18, 8))
keys = [f"Q{q}{s}" for q in range(1, 31) for s in ('a','b')]
for ax, k in zip(axes.flat, keys):
    ax.imshow(cv2.cvtColor(answer_areas[k], cv2.COLOR_BGR2RGB))
    ax.set_title(k, fontsize=7)
    ax.axis('off')
plt.tight_layout()
plt.show()


## Cell 9 - Handwriting preprocessing

CLAHE (local contrast) -> adaptive threshold (handles the pink background
better than a single global Otsu threshold) -> light morphological opening
to clear paper-texture speckle, WITHOUT removing real stroke pixels.


In [ ]:

def binarize_cell(cell_bgr):
    gray = cv2.cvtColor(cell_bgr, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(4, 4))
    gray = clahe.apply(gray)
    binary = cv2.adaptiveThreshold(
        gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV,
        blockSize=21, C=8,
    )
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2, 2))
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
    return binary

def get_clean_ink_components(cell_bgr):
    """Binarize, then drop components that are thin/tall border-line
    artifacts (ruling lines bleeding into the crop) rather than real ink -
    identified by aspect ratio + fill ratio, NOT by raw size, so small but
    genuine digit strokes are kept intact (per your instruction not to
    over-filter)."""
    binary = binarize_cell(cell_bgr)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    clean = np.zeros_like(binary)
    boxes = []
    for i in range(1, n):
        h = stats[i, cv2.CC_STAT_HEIGHT]; w = stats[i, cv2.CC_STAT_WIDTH]
        area = stats[i, cv2.CC_STAT_AREA]
        aspect = h / w if w > 0 else 999
        fill = area / (h * w) if h * w > 0 else 0
        is_border_artifact = (aspect > 2.5 and fill > 0.5) or w <= 2
        if is_border_artifact:
            continue
        clean[labels == i] = 255
        boxes.append((stats[i, cv2.CC_STAT_LEFT], stats[i, cv2.CC_STAT_TOP], w, h))
    return clean, boxes

print("binarize_cell() and get_clean_ink_components() ready")


## Cell 10 - Blank / written detection

Uses foreground density AFTER removing border-line artifacts (Cell 9),
not a single hardcoded ratio. Measured on a real sheet: the max density
among confirmed-blank cells and the min density among confirmed-written
cells were 0.053 and 0.053 respectively - i.e. genuinely overlapping at
that boundary. Rather than fake a clean split, this uses a 3-way decision
with an UNCERTAIN buffer zone that gets flagged for manual review instead
of guessed.


In [ ]:

BLANK_THRESHOLD = 0.04     # below this: confidently BLANK
WRITTEN_THRESHOLD = 0.09   # above this: confidently WRITTEN
# between the two: UNCERTAIN -> needs_review, still attempt OCR but flag it

def classify_blank_written(cell_bgr):
    clean, boxes = get_clean_ink_components(cell_bgr)
    density = clean.sum() / 255 / clean.size
    if density < BLANK_THRESHOLD:
        status = "BLANK"
    elif density > WRITTEN_THRESHOLD:
        status = "WRITTEN"
    else:
        status = "UNCERTAIN"
    return status, density

# Quick check across all 60 cells
for k in list(answer_areas.keys())[:6]:
    status, density = classify_blank_written(answer_areas[k])
    print(f"{k}: {status} (density={density:.4f})")


## Cell 11 - Pretrained handwritten digit recognition

**Measured, not assumed**: EasyOCR and Tesseract were both tested first
against the 5 known digits on this sheet and did poorly (EasyOCR 1/5 at
14% confidence, Tesseract 0/5 - both are built for finding/reading
multi-character text, not isolated single handwritten digits). A CNN
trained on MNIST, using MNIST's own standard digit-centering preprocessing,
did meaningfully better: 3/5 with 80-100% confidence. That's what's used
below. It downloads real MNIST from a GitHub mirror (the official
yann.lecun.com / S3 hosts are blocked in this sandbox - if that's also
blocked for you, torchvision.datasets.MNIST(download=True) is the normal
path).


In [ ]:

import urllib.request, gzip, shutil

MNIST_DIR = "mnist_raw"
os.makedirs(MNIST_DIR, exist_ok=True)
MNIST_FILES = ["train-images-idx3-ubyte", "train-labels-idx1-ubyte",
               "t10k-images-idx3-ubyte", "t10k-labels-idx1-ubyte"]

for fname in MNIST_FILES:
    gz_path = os.path.join(MNIST_DIR, fname + ".gz")
    out_path = os.path.join(MNIST_DIR, fname)
    if not os.path.exists(out_path):
        url = f"https://raw.githubusercontent.com/fgnt/mnist/master/{fname}.gz"
        urllib.request.urlretrieve(url, gz_path)
        with gzip.open(gz_path, 'rb') as f_in, open(out_path, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
print("MNIST files ready:", os.listdir(MNIST_DIR))

def load_idx_images(path):
    with open(path, 'rb') as f:
        _, num, rows, cols = struct.unpack('>IIII', f.read(16))
        return np.frombuffer(f.read(), dtype=np.uint8).reshape(num, rows, cols)

def load_idx_labels(path):
    with open(path, 'rb') as f:
        _, num = struct.unpack('>II', f.read(8))
        return np.frombuffer(f.read(), dtype=np.uint8)

train_images = load_idx_images(f"{MNIST_DIR}/train-images-idx3-ubyte")
train_labels = load_idx_labels(f"{MNIST_DIR}/train-labels-idx1-ubyte")
test_images = load_idx_images(f"{MNIST_DIR}/t10k-images-idx3-ubyte")
test_labels = load_idx_labels(f"{MNIST_DIR}/t10k-labels-idx1-ubyte")

def to_tensor(images, labels):
    x = torch.tensor(images, dtype=torch.float32).unsqueeze(1) / 255.0
    x = (x - 0.1307) / 0.3081
    return x, torch.tensor(labels, dtype=torch.long)

xtr, ytr = to_tensor(train_images, train_labels)
xte, yte = to_tensor(test_images, test_labels)
train_loader = DataLoader(TensorDataset(xtr, ytr), batch_size=256, shuffle=True)

class DigitCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.fc1 = nn.Linear(32*7*7, 128)
        self.fc2 = nn.Linear(128, 10)
    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

digit_model = DigitCNN()
opt = torch.optim.Adam(digit_model.parameters(), lr=1e-3)
digit_model.train()
for epoch in range(3):
    total_loss = 0
    for imgs, labels in train_loader:
        opt.zero_grad()
        loss = F.cross_entropy(digit_model(imgs), labels)
        loss.backward(); opt.step()
        total_loss += loss.item()
    print(f"epoch {epoch+1} loss {total_loss/len(train_loader):.4f}")

digit_model.eval()
with torch.no_grad():
    acc = (digit_model(xte).argmax(1) == yte).float().mean().item()
print(f"MNIST test accuracy: {acc*100:.2f}%")
torch.save(digit_model.state_dict(), "digit_cnn.pt")

def mnist_style_prepare(cell_bgr):
    """Standard MNIST preprocessing: tight-crop to ink bbox, resize
    longest side to 20px preserving aspect ratio, paste into a 28x28
    canvas, then shift so the center of mass sits at the image center."""
    clean, boxes = get_clean_ink_components(cell_bgr)
    if not boxes:
        return None
    x0 = min(b[0] for b in boxes); y0 = min(b[1] for b in boxes)
    x1 = max(b[0]+b[2] for b in boxes); y1 = max(b[1]+b[3] for b in boxes)
    digit = clean[y0:y1, x0:x1]
    h, w = digit.shape
    if h == 0 or w == 0:
        return None
    scale = 20.0 / max(h, w)
    new_h, new_w = max(1, int(h*scale)), max(1, int(w*scale))
    digit_resized = cv2.resize(digit, (new_w, new_h), interpolation=cv2.INTER_AREA)
    canvas = np.zeros((28, 28), dtype=np.uint8)
    y_off, x_off = (28-new_h)//2, (28-new_w)//2
    canvas[y_off:y_off+new_h, x_off:x_off+new_w] = digit_resized
    cy, cx = ndimage.center_of_mass(canvas)
    if not (np.isnan(cy) or np.isnan(cx)):
        canvas = ndimage.shift(canvas, (int(round(14-cy)), int(round(14-cx))), mode='constant', cval=0)
    return canvas

def recognize_digit(cell_bgr):
    """Returns (predicted_digit_or_None, confidence_0_100)."""
    canvas = mnist_style_prepare(cell_bgr)
    if canvas is None:
        return None, 0.0
    x = torch.tensor(canvas, dtype=torch.float32).unsqueeze(0).unsqueeze(0) / 255.0
    x = (x - 0.1307) / 0.3081
    with torch.no_grad():
        probs = F.softmax(digit_model(x), dim=1)
        pred = probs.argmax(1).item()
        conf = probs.max().item() * 100
    return pred, conf

print("recognize_digit() ready")


## Cell 12 - Recognition test against known digits

Ground truth read directly off the uploaded sheet (booklet AC201389):
Q1a=4, Q2b=6, Q3a=2, Q4b=8, Q5a=6. Everything else on this particular
sheet is blank.


In [ ]:

KNOWN_DIGITS = {"Q1a": 4, "Q2b": 6, "Q3a": 2, "Q4b": 8, "Q5a": 6}

print(f"{'ROI':<6}{'actual':>7}{'predicted':>11}{'confidence':>12}  correct")
correct = 0
for roi_name, actual in KNOWN_DIGITS.items():
    pred, conf = recognize_digit(answer_areas[roi_name])
    ok = (pred == actual)
    correct += ok
    print(f"{roi_name:<6}{actual:>7}{str(pred):>11}{conf:>11.1f}%  {'YES' if ok else 'no'}")
print(f"\nAccuracy: {correct}/{len(KNOWN_DIGITS)} ({correct/len(KNOWN_DIGITS)*100:.0f}%)")
print()
print("Honest takeaway: this beats Tesseract (0/5) and EasyOCR (1/5) on the")
print("same crops, but 60% is not reliable enough to trust unattended.")
print("Every prediction should still go through the review dashboard.")
print("The single biggest accuracy lever from here is fine-tuning this CNN")
print("on a batch of YOUR OWN booklets' handwriting rather than stock MNIST,")
print("since MNIST digits are blocky/centered and real ballpoint handwriting")
print("on a ruled form looks meaningfully different.")


## Cell 13 - Subtotal / grand total recognition (multi-digit)

Subtotal/grand total can be 2+ digits (e.g. "26"). Segments the cleaned
ink mask into separate connected components left-to-right and runs the
same CNN on each digit individually, rather than asking a single-character
pipeline to read a multi-digit string.


In [ ]:

def recognize_multidigit(cell_bgr):
    clean, boxes = get_clean_ink_components(cell_bgr)
    if not boxes:
        return None, []
    boxes_sorted = sorted(boxes, key=lambda b: b[0])  # left to right
    digits, confs = [], []
    for (x, y, w, h) in boxes_sorted:
        digit_crop_mask = clean[y:y+h, x:x+w]
        # reuse the same centering pipeline, but on this single component
        scale = 20.0 / max(h, w)
        new_h, new_w = max(1, int(h*scale)), max(1, int(w*scale))
        resized = cv2.resize(digit_crop_mask, (new_w, new_h), interpolation=cv2.INTER_AREA)
        canvas = np.zeros((28, 28), dtype=np.uint8)
        y_off, x_off = (28-new_h)//2, (28-new_w)//2
        canvas[y_off:y_off+new_h, x_off:x_off+new_w] = resized
        cy, cx = ndimage.center_of_mass(canvas)
        if not (np.isnan(cy) or np.isnan(cx)):
            canvas = ndimage.shift(canvas, (int(round(14-cy)), int(round(14-cx))), mode='constant', cval=0)
        xt = torch.tensor(canvas, dtype=torch.float32).unsqueeze(0).unsqueeze(0) / 255.0
        xt = (xt - 0.1307) / 0.3081
        with torch.no_grad():
            probs = F.softmax(digit_model(xt), dim=1)
            digits.append(str(probs.argmax(1).item()))
            confs.append(probs.max().item() * 100)
    value = int("".join(digits)) if digits else None
    return value, confs

for block in (1, 2, 3):
    val, confs = recognize_multidigit(rois[f"subtotal_block{block}"])
    print(f"subtotal_block{block}: value={val}, per-digit confidences={confs}")

gt_val, gt_confs = recognize_multidigit(rois["grand_total"])
print(f"grand_total: value={gt_val}, per-digit confidences={gt_confs}")


## Cell 14 - Question-wise calculation (Qn = a + b, blank = 0)

In [ ]:

def compute_question_marks(recognized):
    """recognized: dict like {'Q1a': (4, 90.0, 'WRITTEN'), 'Q1b': (None, 0, 'BLANK'), ...}
    Returns per-question a/b/total dict, arithmetic only - never asks the
    model to add anything."""
    questions = {}
    for qnum in range(1, 31):
        a_val = recognized.get(f"Q{qnum}a", (None,))[0]
        b_val = recognized.get(f"Q{qnum}b", (None,))[0]
        total = (a_val or 0) + (b_val or 0)
        questions[f"Q{qnum}"] = {"a": a_val, "b": b_val, "total": total}
    return questions

print("compute_question_marks() ready")


## Cell 15 - Checksum / validation

Pure comparison, never invents a digit to force agreement.


In [ ]:

def validate_totals(questions, written_subtotal, written_grand_total):
    calculated_total = sum(q["total"] for q in questions.values())
    verified_subtotal = (written_subtotal is not None and written_subtotal == calculated_total)
    verified_grand_total = (written_grand_total is not None and written_grand_total == calculated_total)
    return calculated_total, verified_subtotal, verified_grand_total

print("validate_totals() ready")


## Cell 16 - Complete process_marksheet() function

Ties every stage together. Booklet number uses plain Tesseract (it's
PRINTED text, not handwriting - a normal OCR job, not a job for the digit
CNN).


In [ ]:

def process_marksheet(image_path, max_marks_per_question=None):
    image = cv2.imread(image_path)
    assert image is not None, f"Could not read {image_path}"
    aligned_img = align_to_template(image)
    rois_ = extract_rois(aligned_img)

    # Booklet number: printed text -> Tesseract, not the digit CNN.
    booklet_text = pytesseract.image_to_string(
        rois_["booklet"],
        config="--oem 3 --psm 7 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"
    ).strip().upper().replace(" ", "")

    recognized = {}
    for qnum in range(1, 31):
        for sub in ("a", "b"):
            key = f"Q{qnum}{sub}"
            cell = extract_answer_area(rois_[key])
            status, density = classify_blank_written(cell)
            if status == "BLANK":
                recognized[key] = (None, 0.0, status)
            else:
                pred, conf = recognize_digit(cell)
                # UNCERTAIN cells still get a prediction, but flagged.
                recognized[key] = (pred, conf, status)

    questions = compute_question_marks(recognized)

    subtotal_val, _ = recognize_multidigit(rois_["subtotal_block1"])  # adjust per block as needed
    grand_total_val, _ = recognize_multidigit(rois_["grand_total"])

    calculated_total, verified_subtotal, verified_grand_total = validate_totals(
        questions, subtotal_val, grand_total_val
    )

    needs_review = [
        k for k, (pred, conf, status) in recognized.items()
        if status == "UNCERTAIN" or (status == "WRITTEN" and conf < 70)
    ]

    result = {
        "booklet": booklet_text,
        "questions": questions,
        "calculated_total": calculated_total,
        "written_subtotal": subtotal_val,
        "written_grand_total": grand_total_val,
        "verified_subtotal": verified_subtotal,
        "verified_grand_total": verified_grand_total,
        "overall_status": "VERIFIED" if (verified_subtotal or verified_grand_total) else "NEEDS_REVIEW",
        "needs_review_cells": needs_review,
    }
    if max_marks_per_question:
        violations = [
            qk for qk, q in questions.items()
            if q["total"] > max_marks_per_question.get(qk, 10)
        ]
        result["max_marks_violations"] = violations
    return result, recognized, aligned_img

print("process_marksheet() ready")


## Cell 17 - Final JSON result (run on the actual uploaded sheet)

In [ ]:

import json

result, recognized, _ = process_marksheet("aligned.png")  # already-aligned from Cell 4
print(json.dumps(result, indent=2, default=str))


## Cell 18 - Final visualization / report

For each nonblank question: original ROI -> clean answer area -> model
input -> prediction -> confidence. Blank cells shown separately.


In [ ]:

nonblank = {k: v for k, v in recognized.items() if v[2] != "BLANK"}
n = max(1, len(nonblank))
fig, axes = plt.subplots(n, 4, figsize=(10, 2.2*n))
if n == 1:
    axes = axes.reshape(1, -1)

for row, (key, (pred, conf, status)) in enumerate(nonblank.items()):
    raw = rois[key]
    area = extract_answer_area(raw)
    clean, _ = get_clean_ink_components(area)
    model_input = mnist_style_prepare(area)

    axes[row][0].imshow(cv2.cvtColor(raw, cv2.COLOR_BGR2RGB)); axes[row][0].set_title(f"{key} raw", fontsize=8)
    axes[row][1].imshow(area[..., ::-1]); axes[row][1].set_title("answer area", fontsize=8)
    axes[row][2].imshow(clean, cmap='gray'); axes[row][2].set_title("clean ink", fontsize=8)
    if model_input is not None:
        axes[row][3].imshow(model_input, cmap='gray')
    axes[row][3].set_title(f"pred={pred} ({conf:.0f}%) [{status}]", fontsize=8)
    for c in range(4):
        axes[row][c].axis('off')

plt.tight_layout()
plt.show()

blanks = [k for k, v in recognized.items() if v[2] == "BLANK"]
print(f"\n{len(blanks)} cells classified BLANK (of 60 total question sub-cells).")


---
# v2 updates - applied after testing against your real sheet

Three real, measured fixes below, based on actually running v1 end-to-end
and finding genuine bugs/weaknesses rather than assuming the design was
right:

1. **Booklet, subtotal, and grand-total box coordinates were wrong.** v1's
   guessed boxes clipped the booklet number against the "SI.No." label
   text and used the narrow per-question answer-row height for the taller
   subtotal digits. Both are now re-measured against your actual sheet.
2. **Multi-character fields need a sequence reader, not per-digit
   segmentation.** Your handwritten "26" subtotal has a joined/cursive "2"
   and "6" - connected-component segmentation produces exactly ONE blob,
   not two, so per-digit classification cannot split it. EasyOCR read the
   whole crop correctly (**26**, 71% confidence) with no segmentation
   needed. Same story for the printed booklet number: EasyOCR got
   **AC201389 at 97.3% confidence** (Tesseract's own single-line attempt
   got one digit wrong: AC201383). **New rule: single isolated handwritten
   digit cells use the digit CNN; multi-character fields (booklet,
   subtotal, grand total) use EasyOCR.**
3. **Tested the "BLANK as an explicit 12th class" idea from your source
   and it's a real trade-off, not a clean win here**: training one 11-class
   softmax (digits 0-9 + BLANK, using your sheet's 55 real blank cells
   augmented to 1,350 samples) correctly called 8/10 held-out real blanks
   BLANK - genuinely better than the old density-threshold heuristic's
   ambiguous boundary case. But it also *regressed* single-digit accuracy
   from 3/5 to 2/5 on the same known digits, because one softmax now has
   to do two different jobs (density estimation for blank-ness feeds
   the same 128-d feature layer as the actual digit shape decision).
   **Kept the two decisions decoupled instead**: a density-based
   blank/written/uncertain gate first (Cell 10, unchanged), then the
   pure single-purpose MNIST-only digit CNN only on cells that pass the
   gate. That is what Cell 16's `process_marksheet()` uses below.


In [ ]:

# --- Corrected template boxes (measured against the real uploaded sheet) ---
TEMPLATE["booklet_box"] = {"x0": 815, "y0": 75, "x1": 1085, "y1": 135}
TEMPLATE["grand_total_box"] = {"x0": 990, "y0": 478, "x1": 1075, "y1": 515}
# Subtotal boxes need a TALLER y-band than the narrow per-question answer
# row - the handwritten subtotal sits across roughly the row block's full
# height, not just the ~37px answer sub-row.
TEMPLATE["subtotal_bands"] = {1: (240, 320), 2: (316, 396), 3: (392, 472)}

def extract_rois_v2(aligned_image):
    rois_ = extract_rois(aligned_image)
    xb = TEMPLATE["x_boundaries"]
    bb = TEMPLATE["booklet_box"]
    gt = TEMPLATE["grand_total_box"]
    rois_["booklet"] = aligned_image[bb["y0"]:bb["y1"], bb["x0"]:bb["x1"]]
    rois_["grand_total"] = aligned_image[gt["y0"]:gt["y1"], gt["x0"]:gt["x1"]]
    for block, (y1, y2) in TEMPLATE["subtotal_bands"].items():
        rois_[f"subtotal_block{block}"] = aligned_image[y1:y2, xb[20]:xb[21]]
    return rois_

print("extract_rois_v2() ready with corrected boxes")


In [ ]:

import easyocr
_easyocr_reader = easyocr.Reader(['en'], gpu=False, verbose=False)

def read_multichar_field(cell_bgr, allowlist):
    """For booklet number / subtotal / grand total: multi-character
    fields where EasyOCR's sequence model outperformed per-digit
    segmentation in real testing (handles joined/cursive characters that
    connected-component segmentation cannot split)."""
    gray = cv2.cvtColor(cell_bgr, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, None, fx=3, fy=3, interpolation=cv2.INTER_CUBIC)
    results = _easyocr_reader.readtext(gray, allowlist=allowlist, detail=1)
    if not results:
        return None, 0.0
    text = "".join(r[1] for r in results)
    conf = max(r[2] for r in results) * 100
    return text, conf

# Real test against your sheet:
test_rois = extract_rois_v2(aligned)
booklet_text, booklet_conf = read_multichar_field(test_rois["booklet"], "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789")
subtotal1_val, subtotal1_conf = read_multichar_field(test_rois["subtotal_block1"], "0123456789")
print(f"booklet: {booklet_text} ({booklet_conf:.1f}%)")
print(f"subtotal_block1: {subtotal1_val} ({subtotal1_conf:.1f}%)")


## Checksum reconciliation with confidence-based flagging

Per your source's point 10/11: if the calculated total doesn't match the
written subtotal, don't just fail - find the lowest-confidence cell(s),
check whether swapping in their 2nd-best prediction resolves the gap, and
if not, flag only those cells for human review rather than the whole
sheet or (worse) silently forcing agreement.


In [ ]:

def reconcile_checksum(cell_predictions, written_total):
    """cell_predictions: dict of {roi_id: (value, confidence_0_100, top2_value_or_None)}
    Never invents a digit to force a match - only reports what it tried
    and whether it worked, so a human makes the final call either way."""
    calculated = sum(v for v, c, t2 in cell_predictions.values() if v is not None)
    if written_total is not None and calculated == written_total:
        return {"status": "AUTO_VERIFIED", "calculated": calculated, "flagged": []}

    delta = None if written_total is None else written_total - calculated
    sorted_by_conf = sorted(cell_predictions.items(), key=lambda kv: kv[1][1])

    if delta is not None:
        for roi_id, (val, conf, top2) in sorted_by_conf:
            if top2 is not None and val is not None:
                if (calculated - val + top2) == written_total:
                    return {
                        "status": "RESOLVED_BY_SECOND_GUESS",
                        "calculated": calculated,
                        "suggested_fix": {roi_id: top2},
                        "flagged": [roi_id],
                    }

    flagged = [roi_id for roi_id, (val, conf, top2) in sorted_by_conf if conf < 70][:3]
    return {"status": "NEEDS_HUMAN_REVIEW", "calculated": calculated, "flagged": flagged}

print("reconcile_checksum() ready")
